# GAIPO: Graph Artificial Intelligence for Pediatric Oncology

GAIPO is a modular, configuration-driven platform for integrating pediatric oncology clinical and multi-omics data, constructing patient-similarity graphs, training graph AI models, and performing explainable downstream analyses. The workflow is designed around the Childhood Cancer Data Initiative (CCDI) ecosystem and GDC-compatible data resources.

GAIPO organizes the analysis into seven reproducible stages:

1. **Data fetching** — discover subject and sample identifiers through the CCDI Data Federation.
2. **Data extraction** — retrieve clinical and molecular data from cBioPortal-compatible APIs.
3. **Data modeling** — harmonize source data into a GAIPO Data Model (GDM) aligned with GDC concepts.
4. **Data processing** — perform quality control, feature selection, scaling, and train/test preparation.
5. **Graph construction** — build leakage-aware patient-similarity graphs for each modality.
6. **Graph AI modeling** — train or evaluate multi-view graph neural networks.
7. **Post-analysis** — interpret features, evaluate survival outcomes, and summarize learned representations.

> **Research-use notice:** GAIPO is research software and is not intended for clinical diagnosis, treatment selection, or direct patient care.

## Repository Structure

The following is a simplified view of the repository and generated data artifacts. Some folders are created only after the corresponding pipeline stage runs.

<pre><code>GAIPO/
├── Dockerfile
├── docker-compose.yaml
├── requirements.txt
├── config/
│   ├── pipeline_config.yaml
│   └── gdc_mapping.yaml
├── src/
│   ├── gdcdictionary/
│   │   └── schemas/
│   ├── main.py
│   ├── data_source.py          # Fetch subject and sample IDs from the CCDI Federation
│   ├── extractor.py            # Extract clinical and molecular data from cBioPortal APIs
│   ├── data_model.py           # Build GDC-aligned Parquet tables and DuckDB views
│   ├── processor.py            # Filter, select, scale, and split features
│   ├── graph_constructor.py    # Construct patient-similarity graphs
│   ├── graph_ai_model.py       # Train and evaluate graph AI models
│   └── post_analysis.py        # Interpret models and perform downstream analyses
└── data/
    ├── fetch/
    │   ├── ccdi_api_request/
    │   │   ├── sample_ids/
    │   │   │   └── {cancer_type | study_id}.csv
    │   │   └── subject_ids/
    │   │       └── {cancer_type | study_id}.csv
    │   └── cbioportal_api_request/
    │       └── {cancer_type | study_id}/
    │           ├── clinical_patients.csv
    │           ├── clinical_samples.csv
    │           └── {modality}.tsv
    └── gdm/                    # GAIPO Data Model
        ├── clinical/
        ├── nodes/
        ├── indexes/
        ├── files/
        ├── features/
        ├── filtered/
        └── graphs/
</code></pre>

## Tutorial Roadmap

This notebook has two goals:

1. Explain the purpose, inputs, outputs, and configuration of every GAIPO stage.
2. Provide safe, reproducible examples for discovering studies and running the pipeline.


## 1. Prerequisites and Study Discovery

### 1.1 Prerequisites

Run the notebook from the GAIPO repository root. For local execution, create a virtual environment and install the project dependencies:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

Docker Compose is recommended when you want a reproducible environment without managing local Python dependencies.

### 1.2 cBioPortal study IDs

The public cBioPortal API can be queried without an API token. The following cell searches study names, descriptions, and cancer-type fields for candidate glioma and Wilms tumor studies. Review the returned studies before adding a `studyId` to `config/pipeline_config.yaml`; keyword matching identifies candidates but does not guarantee that a study has the required clinical endpoints or molecular profiles.


In [ ]:
# Discover candidate glioma and Wilms tumor studies through the public cBioPortal API.
import requests

CBIOPORTAL_API = "https://www.cbioportal.org/api"
HEADERS = {"Accept": "application/json"}

response = requests.get(
    f"{CBIOPORTAL_API}/studies",
    headers=HEADERS,
    timeout=60,
)
response.raise_for_status()
public_studies = response.json()


def matches_any_keyword(study, keywords):
    """Return True when a keyword occurs in a study's searchable metadata."""
    searchable_text = " ".join(
        str(study.get(field, ""))
        for field in ("studyId", "name", "description", "cancerType")
    ).lower()
    return any(keyword.lower() in searchable_text for keyword in keywords)


glioma_keywords = ("pediatric glioma", "high-grade glioma", "low-grade glioma")
wilms_keywords = ("wilms", "nephroblastoma")

glioma_studies = [
    study for study in public_studies
    if matches_any_keyword(study, glioma_keywords)
]
wilms_studies = [
    study for study in public_studies
    if matches_any_keyword(study, wilms_keywords)
]

print("Glioma candidates:")
for study in glioma_studies:
    print(f"  {study['studyId']}: {study['name']}")

print("\nWilms tumor candidates:")
for study in wilms_studies:
    print(f"  {study['studyId']}: {study['name']}")


### 1.3 PedcBioPortal study IDs

PedcBioPortal requires an API token. Download a token from your PedcBioPortal profile and expose it only through an environment variable. Never place the token directly in the notebook, configuration files, source code, screenshots, or Git history.

In a terminal, set the variable before starting Jupyter:

```bash
export PEDCBIO_TOKEN="your-token"
```

If a real token has previously been committed or shared, revoke or rotate it before continuing. The next cell reads the token from `PEDCBIO_TOKEN` and searches for PBTA, Wilms tumor, and nephroblastoma studies.


In [ ]:
# Discover candidate studies through the authenticated PedcBioPortal API.
import os

from bravado.client import SwaggerClient
from bravado.requests_client import RequestsClient

PEDCBIOPORTAL_BASE = "https://pedcbioportal.kidsfirstdrc.org"
SWAGGER_URL = f"{PEDCBIOPORTAL_BASE}/api/v2/api-docs"
pedcbio_token = os.getenv("PEDCBIO_TOKEN")

if not pedcbio_token:
    raise RuntimeError(
        "PEDCBIO_TOKEN is not set. Export it in your shell before running this cell."
    )

http_client = RequestsClient()
http_client.session.headers.update(
    {
        "Authorization": f"Bearer {pedcbio_token}",
        "Accept": "application/json",
    }
)

pedcbio_client = SwaggerClient.from_url(
    SWAGGER_URL,
    http_client=http_client,
    config={
        "validate_requests": False,
        "validate_responses": False,
        "validate_swagger_spec": False,
    },
)

pedcbio_studies = pedcbio_client.Studies.getAllStudiesUsingGET().result()


def object_matches(study, keywords):
    searchable_text = f"{study.studyId} {study.name or ''}".lower()
    return any(keyword.lower() in searchable_text for keyword in keywords)


pbta_studies = [
    study for study in pedcbio_studies
    if object_matches(study, ("pbta", "pediatric brain tumor atlas"))
]
wilms_studies = [
    study for study in pedcbio_studies
    if object_matches(study, ("wilms", "nephroblastoma"))
]

print("PBTA candidates:")
for study in pbta_studies:
    print(f"  {study.studyId}: {study.name}")

print("\nWilms tumor candidates:")
for study in wilms_studies:
    print(f"  {study.studyId}: {study.name}")


## 2. Stage 1 — Fetch Subject and Sample IDs

The `data_fetch` stage queries the CCDI Data Federation, classifies returned diagnoses into configured cancer types, and creates identifier manifests for downstream extraction and joining.

### What the stage does

1. Builds an active cancer-type vocabulary from `default_type_terms` and `custom_type_terms`.
2. Restricts processing to entries in `cancer_type_interest`.
3. Pages through the `/subject-diagnosis` and `/sample-diagnosis` endpoints.
4. Classifies records using word-boundary regular expressions applied to associated diagnoses.
5. Writes cancer-specific subject and sample CSV files.
6. Creates or updates `idmap.parquet` for downstream graph and clinical-data joins.

### Representative configuration

```yaml
Data:
  ccdi_sample_diagnosis: https://federation.ccdi.cancer.gov/api/v1/sample-diagnosis
  ccdi_subject_diagnosis: https://federation.ccdi.cancer.gov/api/v1/subject-diagnosis

cancer_type_interest:
  - Wilms Tumor

default_type_terms:
  Wilms Tumor:
    - wilms
    - nephroblastoma

custom_type_terms: {}
target_source: null       # Or PCDC, Treehouse, StJude, KidsFirst, or ccdi-ecDNA
per_page: 100
timeout: 60
retry_status: [429, 500, 502, 503, 504]

studyId:
  Wilms Tumor: wt_target_2018_pub
```

### Outputs

```text
data/fetch/ccdi_api_request/
├── subject_ids/<cancer_type>.csv
└── sample_ids/<cancer_type>.csv

data/gdm/indexes/
└── idmap.parquet
```

Typical identifier columns include `source`, `subject_id`, `sample_id`, `diagnoses`, `study_id`, `node_id_subject`, and `node_id_sample` as applicable.

### Run the stage

```bash
python -m src.main --call data_fetch
```

### Notes

- Set `target_source` when a cohort should be restricted to one CCDI Federation backend.
- Diagnosis classification depends on the returned `associated_diagnoses` values and the configured vocabulary; inspect unmatched or ambiguous records.
- HTTP 429 and configured server errors are retried with exponential backoff.
- The identifier map is deduplicated by sample ID and is intended to be safely updated across repeated runs.


## 3. Stage 2 — Extract Clinical and Molecular Data

The `data_extract` stage uses configured study IDs and subject/sample manifests to retrieve processed clinical and molecular profiles from cBioPortal-compatible APIs.

### Inputs

- Cancer types and study IDs from `config/pipeline_config.yaml`.
- Subject and sample manifests produced by `data_fetch`.
- A PedcBioPortal token when an authenticated pediatric portal is selected.
- Requested molecular profiles, such as mRNA, miRNA, methylation, or copy-number alteration data.

### Typical outputs

```text
data/fetch/cbioportal_api_request/<Cancer_Type>/
├── clinical_patients.csv
├── clinical_samples.csv
└── <molecular_profile>.tsv
```

Feature stores and their indexes may also be registered under:

```text
data/gdm/features/
├── mrna.zarr
├── mirna.zarr
└── methylation.zarr

data/gdm/indexes/
└── <modality>_{rows,cols}.parquet
```

Only files supported by the selected study and enabled in the configuration are created.

### Run the stage

```bash
python -m src.main --call data_extract
```

### Validation checks

- Confirm that returned sample IDs overlap the CCDI manifest or document why a portal-specific cohort is being used.
- Check the number of patients, samples, and molecular profiles before modeling.
- Verify feature orientation: rows should represent samples and columns should represent molecular features before processing.
- Treat empty responses as a study/profile compatibility issue rather than silently continuing.


## 4. Stage 3 — Build the GAIPO Data Model

GAIPO maps cBioPortal clinical and molecular data into a GDC-aligned representation and exposes the resulting Parquet files through DuckDB views. This layer provides consistent identifiers and table semantics across cohorts and modalities.

### What the stage does

1. Loads local GDC dictionary YAML schemas and normalizes references and anchors.
2. Projects source clinical tables into GDC-like case, sample, demographic, and diagnosis nodes.
3. Builds sample-to-case and demographic-to-case link tables.
4. Maintains an identifier map for clinical, molecular, and graph joins.
5. Registers available Zarr feature stores as file nodes.
6. Creates DuckDB temporary views for cohort queries.

### Inputs

```text
data/fetch/cbioportal_api_request/<Cancer_Type>/
├── clinical_patients.csv
└── clinical_samples.csv

data/gdm/features/
└── <modality>.zarr

data/gdm/indexes/
└── <modality>_{rows,cols}.parquet
```

Mapping rules are defined in `config/gdc_mapping.yaml`, including the project ID, schema directory, GDM root, disease and primary-site mappings, sample suffixes, and preservation mappings.

### Outputs

```text
data/gdm/
├── clinical/
│   ├── clinical_subjects.parquet
│   └── clinical_samples.parquet
├── nodes/
│   ├── case.parquet
│   ├── sample.parquet
│   ├── demographic.parquet
│   ├── diagnosis.parquet
│   └── nodes.parquet
├── indexes/
│   ├── idmap.parquet
│   ├── links_sample_case.parquet
│   └── links_demographic_case.parquet
├── files/
│   ├── file.parquet
│   └── file_full.parquet
└── features/
    ├── mrna.zarr
    ├── mirna.zarr
    └── methylation.zarr
```

### DuckDB temporary views

| View | Backing data |
| --- | --- |
| `subjects` | `clinical/clinical_subjects.parquet` |
| `samples` | `clinical/clinical_samples.parquet` |
| `gdc_case` | `nodes/case.parquet` |
| `gdc_demographic` | `nodes/demographic.parquet` |
| `gdc_diagnosis` | `nodes/diagnosis.parquet` |
| `links_sample_case` | `indexes/links_sample_case.parquet` |
| `links_demographic_case` | `indexes/links_demographic_case.parquet` |
| `idmap` | `indexes/idmap.parquet` |

The view name `gdc_case` avoids the SQL reserved word `case`.

### Validation modes and environment variables

| Setting | Purpose |
| --- | --- |
| `DM_CONFIG` | Alternative data-model configuration path |
| `CONFIG_PATH` | Pipeline configuration path |
| `DM_GDC_STRICT=1` | Enforce strict GDC dictionary requirements |
| `DM_DUCKDB_MODE` | `memory`, `ro`, or `rw` |
| `DM_DUCKDB_PATH` | Persistent DuckDB path when using read/write mode |
| `DM_DUCKDB_RETRIES` | Lock-retry count for read/write mode |
| `DM_DUCKDB_RETRY_SLEEP` | Seconds between lock retries |

Lite validation is appropriate during iterative development. Strict validation should be used when mappings are sufficiently complete for release-quality data.

### Run the stage

```bash
python -m src.main --call data_model
```


### 4.1 Visualize a YAML Data Model

The helper below converts a YAML mapping with `Nodes` and `Relationships` sections into a Graphviz diagram. Python's `graphviz` package and a system Graphviz installation are required.


In [ ]:
from html import escape

import graphviz
import yaml


def plot_model_from_mapping(model, width=12, height=8, rankdir="TB"):
    """Create a Graphviz diagram from a GAIPO-style YAML model mapping."""
    dot = graphviz.Digraph("DataModel", comment="GAIPO data-model relationships")
    dot.attr(size=f"{width},{height}!", rankdir=rankdir, splines="ortho", nodesep="1.0")
    dot.attr("node", shape="plain")
    dot.attr("edge", fontsize="12", fontcolor="gray40")

    nodes = model.get("Nodes", {})
    for node_name, node_details in nodes.items():
        safe_name = escape(str(node_name))
        rows = [
            '<TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" CELLPADDING="4">',
            f'<TR><TD BGCOLOR="#a9d0f5"><B>{safe_name}</B></TD></TR>',
        ]
        for prop in node_details.get("Props", []) or []:
            rows.append(f'<TR><TD ALIGN="LEFT">{escape(str(prop))}</TD></TR>')
        rows.append("</TABLE>")
        dot.node(str(node_name), f"<{''.join(rows)}>")

    for relationship_name, relationship in model.get("Relationships", {}).items():
        for link in relationship.get("Ends", []) or []:
            source = link.get("Src")
            target = link.get("Dst")
            if source in nodes and target in nodes:
                edge_label = link.get("Label", relationship_name)
                dot.edge(str(source), str(target), label=str(edge_label))

    return dot


In [ ]:
# Minimal example. Replace this YAML string with a loaded GAIPO/GDC model as needed.
example_yaml = """
Handle: C3DC
Version: v4.0.5
Nodes:
  diagnosis:
    Props: [age_at_diagnosis, diagnosis_id, diagnosis]
  participant:
    Props: [participant_id, race, sex_at_birth]
  sample:
    Props: [sample_id, anatomic_site, sample_tumor_status]
  study:
    Props: [study_id, study_name, dbgap_accession]
  survival:
    Props: [survival_id, last_known_survival_status]
Relationships:
  of_diagnosis:
    Ends:
      - {Src: diagnosis, Dst: participant}
      - {Src: diagnosis, Dst: sample}
  of_participant:
    Ends:
      - {Src: participant, Dst: study}
  of_sample:
    Ends:
      - {Src: sample, Dst: participant}
  of_survival:
    Ends:
      - {Src: survival, Dst: participant}
"""

example_model = yaml.safe_load(example_yaml)
data_model_graph = plot_model_from_mapping(example_model)
data_model_graph

# To save the rendered diagram, uncomment the following line:
# data_model_graph.render("GAIPO_data_model", format="png", cleanup=True)


## 5. Stage 4 — Process Clinical and Multi-omics Features

The `process` stage transforms extracted feature matrices into model-ready training and test inputs. All data-dependent transformations must be fitted using the training set only and then applied unchanged to validation or test data.

### Typical processing sequence

1. Align samples across clinical labels and selected modalities.
2. Remove features with excessive zeros or missing values.
3. Remove constant and low-variance features.
4. Apply configured univariate feature screening, such as ANOVA with Benjamini–Hochberg FDR control.
5. Fit dimensionality reduction when enabled.
6. Scale each modality using parameters estimated from the training cohort.
7. Save aligned train/test matrices, labels, sample IDs, and fitted preprocessing metadata.

### Inputs and outputs

```text
Input:
  data/gdm/features/<modality>.zarr
  data/gdm/indexes/<modality>_{rows,cols}.parquet
  data/gdm/clinical/ and data/gdm/nodes/

Output:
  data/gdm/filtered/<Cancer_Type>/<modality>_train.parquet
  data/gdm/filtered/<Cancer_Type>/<modality>_test.parquet
  aligned labels, sample IDs, and preprocessing metadata
```

### Run the stage

```bash
python -m src.main --call process
```

### Leakage-prevention checklist

- Create patient-level splits before fitting selectors, scalers, or PCA.
- Keep samples from the same subject in a single split.
- Fit every filtering threshold and transformation on training data only.
- Save the fitted feature order and reuse it exactly during inference.
- Report exclusions and missing modalities for each cohort.


## 6. Stage 5 — Construct Patient-similarity Graphs

GAIPO currently builds MOGONET-style patient-similarity graphs for bulk multi-omics data. Spatial single-cell graph constructors are reserved for future implementation.

### Bulk multi-omics graph construction

Each modality is represented as a feature matrix with patients as rows and features as columns. GAIPO then:

1. Computes cosine distances between training patients.
2. Derives an adaptive training radius targeting approximately `edges_per_node` neighbors.
3. Builds the training graph using the training-derived radius.
4. Builds a combined train/test block graph with train-to-test edges but no test-to-test edges.
5. Converts cosine distance to similarity weights using `1 - distance`.
6. Symmetrizes the adjacency matrix, adds self-loops, and row-normalizes it.

Using the training-derived radius for held-out samples prevents the test cohort from influencing graph topology.

### Inputs

```text
data/gdm/filtered/<Cancer_Type>/<modality>_train.parquet
data/gdm/filtered/<Cancer_Type>/<modality>_test.parquet
```

Supported modality names are configuration-dependent; common values include `mrna`, `mirna`, and `methylation`. Synonyms such as `microrna` may be normalized to `mirna`.

### Representative configuration

```yaml
graph:
  edges_per_node: 15
  save_graphml: false

cancer_type_interest:
  - Wilms Tumor

expression_profiles:
  Wilms Tumor:
    - mrna
    - mirna
    - methylation
```

### Outputs

```text
data/gdm/graphs/<Cancer_Type>/
├── <modality>_train.adj.npz
├── <modality>_trte_block.adj.npz
└── <modality>_train.graphml        # Optional
```

Adjacency matrices are SciPy CSR matrices saved with `scipy.sparse.save_npz`.

### Run through the pipeline

```bash
python -m src.main --call graph_construct
```

### Ad-hoc CLI examples

```bash
python src/graph_constructor.py bulk-train \
  --matrix data/gdm/filtered/mrna_train.parquet \
  --format parquet \
  --edges-per-node 15 \
  --out data/gdm/graphs/mrna_train.adj.npz \
  --out-graphml data/gdm/graphs/mrna_train.graphml

python src/graph_constructor.py bulk-split \
  --matrix-train data/gdm/filtered/mrna_train.parquet \
  --matrix-test data/gdm/filtered/mrna_test.parquet \
  --format parquet \
  --edges-per-node 15 \
  --out-train data/gdm/graphs/mrna_train.adj.npz \
  --out-trte data/gdm/graphs/mrna_trte_block.adj.npz
```

Useful helpers include `csr_to_edge_index`, `csr_to_torch_coo`, `csr_to_networkx`, `save_adjacency_npz`, and `save_graph_graphml`.


## 7. Stage 6 — Train and Evaluate Graph AI Models

GAIPO provides a MOGONET-style multi-view GNN adapted to the project's feature and adjacency formats. Modality-specific GCN encoders learn patient representations, and a view-correlation discovery network (VCDN) can fuse predictions across modalities.

### Supported workflows

1. **Train and evaluate** — load aligned train/test features and graphs, train the encoders and classifiers, and report configured metrics.
2. **Pretrained inference** — load saved weights, attach validation patients using the training-derived graph radius, and generate predictions without refitting preprocessing or graph parameters.
3. **Feature interpretation** — quantify predictive contributions using configured attribution or ablation procedures.

Depending on the configured model head, outputs may include class probabilities, survival-risk scores, learned embeddings, model checkpoints, and feature-importance results.

### Required upstream artifacts

- `processor.py`: aligned feature dictionaries, labels or survival endpoints, sample IDs, and saved preprocessing state.
- `graph_constructor.py`: training and train/test block adjacencies built with training-derived parameters.

### Run the stage

```bash
python -m src.main --call graph_ai_model
```

### Evaluation guidance

- Keep the same sample order across features, labels, and adjacency matrices.
- Select hyperparameters without using the held-out test set.
- Report cohort sizes and class/event distributions for every split.
- Use classification metrics appropriate for imbalance, such as balanced accuracy, macro F1, AUROC, and AUPRC.
- For survival tasks, report concordance and time-to-event group comparisons with censoring handled explicitly.


## 8. Stage 7 — Post-analysis and Interpretation

The `post_analysis` stage converts model outputs into interpretable statistical summaries and figures.

Typical analyses include:

- Feature attribution or ablation summaries by modality.
- Biomarker ranking with direction and magnitude of contribution.
- Kaplan–Meier curves and log-rank tests for predefined or model-derived groups.
- Latent-space visualization and hierarchical clustering.
- Export of prediction, embedding, feature-ranking, and survival-result tables.

### Run the stage

```bash
python -m src.main --call post_analysis
```

### Survival-analysis safeguards

- Define the survival time scale and event coding in the configuration and report them with results.
- Derive any risk-group cutoff from the training set, then reuse it unchanged in validation and test cohorts.
- Use a consistent color and legend mapping across train, test, and combined Kaplan–Meier figures.
- Report the number at risk, number of events, hazard-ratio uncertainty when applicable, and log-rank p-value.
- Treat exploratory biomarker rankings as hypotheses requiring external validation.


## 9. Run GAIPO

Run all commands from the repository root.

### Run one stage

```bash
python -m src.main --call data_fetch
python -m src.main --call data_extract
python -m src.main --call data_model
python -m src.main --call process
python -m src.main --call graph_construct
python -m src.main --call graph_ai_model
python -m src.main --call post_analysis
```

### Run selected stages

Provide stage names as a comma-separated list without spaces or shell braces:

```bash
python -m src.main --call data_fetch,data_extract,data_model
python -m src.main --call graph_construct,graph_ai_model,post_analysis
```

### Run through a specified stage

```bash
python -m src.main --until graph_construct
```

### Run the complete pipeline

```bash
python -m src.main --all
```

The full execution order is:

```text
data_fetch → data_extract → data_model → process → graph_construct → graph_ai_model → post_analysis
```

### Docker Compose

```bash
# Optional for recent Docker Compose versions
export COMPOSE_BAKE=true

docker compose build
docker compose run --rm app python -m src.main --all
docker compose run --rm app python -m src.main --call data_fetch,data_extract
docker compose run --rm app python -m src.main --until graph_construct
```

The Compose configuration should mount data, results, and model directories so that generated artifacts persist after the temporary `app` container exits.


## 10. Ad-hoc Data-model and DuckDB Examples

The main pipeline is preferred because it applies configuration consistently. For focused development, the data-model module can also be run directly:

```bash
python -m src.data_model \
  --schemas-dir src/gdcdictionary/schemas \
  --gdm-root data/gdm \
  --cbio-dir data/fetch/cbioportal_api_request/Wilms_Tumor \
  --project-id WILMS_TUMOR_CBIO
```

The following example queries the registered DuckDB views for samples from female participants with a stage III diagnosis:

```bash
PIPELINE_SQL="$(cat <<'SQL'
WITH cohort AS (
    SELECT c.submitter_id AS subject_id
    FROM gdc_case AS c
    LEFT JOIN gdc_diagnosis AS d
        ON d.case_submitter_id = c.submitter_id
    LEFT JOIN links_demographic_case AS ldc
        ON ldc.case_submitter_id = c.submitter_id
    LEFT JOIN gdc_demographic AS demo
        ON demo.submitter_id = ldc.demographic_submitter_id
    WHERE upper(coalesce(d.tumor_stage, '')) LIKE 'III%'
      AND lower(coalesce(demo.sex_at_birth, '')) = 'female'
)
SELECT DISTINCT i.sample_id
FROM cohort AS c
JOIN idmap AS i USING (subject_id)
ORDER BY i.sample_id;
SQL
)"

export PIPELINE_SQL
python -m src.main --call data_query
```


## 11. Reproducibility, Security, and Troubleshooting

### Reproducibility checklist

For each experiment, retain:

- The pipeline and mapping configuration files.
- The source studies, CCDI sources, and cohort manifest.
- Train/validation/test subject IDs.
- Fitted preprocessing and graph-construction parameters.
- Random seeds, package versions, and model checkpoints.
- Evaluation metrics and post-analysis settings.

### Security and data governance

- Keep API tokens in environment variables or an approved secret manager.
- Never commit `.env` files, tokens, credentials, or controlled-data identifiers.
- Follow the data-use agreement and access policy for every source.
- Confirm that exported tables and figures contain no protected or restricted identifiers before sharing.

### Troubleshooting

| Problem | Check |
| --- | --- |
| `No module named src` | Run the command from the repository root and activate the intended environment. |
| A downstream stage cannot find inputs | Run the required upstream stages and verify configured artifact paths. |
| cBioPortal returns no profiles | Confirm the study ID, molecular-profile IDs, permissions, and study availability. |
| PedcBioPortal returns an authentication error | Export a current `PEDCBIO_TOKEN`; do not paste it into the notebook. |
| Docker cannot write outputs | Check host-directory mounts, ownership, and permissions in `docker-compose.yaml`. |
| Sample counts change unexpectedly | Inspect identifier overlap, deduplication, missing modalities, and cohort filters. |
| Validation performance appears implausibly high | Check split integrity and ensure preprocessing and graph parameters were fitted on training data only. |
| Results differ across runs | Fix random seeds and reuse the same configuration, manifests, split, and dependency versions. |

## 12. Recommended Validation Before Release

Before publishing a GAIPO analysis or software release:

1. Run the full pipeline in a clean environment.
2. Confirm that every documented path and CLI name matches the current repository.
3. Validate intermediate sample counts and identifier joins.
4. Test strict GDC validation when mappings are complete.
5. Verify that held-out data did not influence preprocessing, graph construction, model selection, or risk thresholds.
6. Remove credentials, sensitive identifiers, temporary files, and stale notebook outputs.
7. Record the software version and cite the corresponding GAIPO release or manuscript.
